In [9]:
import numpy as np
import glob
from scipy.special import gammaln
import random
from scipy.optimize import minimize
import tskit
import pandas as pd

n = 20  # Number of samples or lineages at start 
# N_e1 = 1500  # Effective population size in interval [0, T1]
# N_e2 = 500  # Effective population size in interval [T1, T2]
# N_e3 = 2500   # Effective population size in interval [T2, infinity]
def get_equal_count_time_bins(path_pattern="/space/s1/KaiyuanLi/Ne_estimate/ratio1/1/arg_nodes_50_*.txt", num_bins=20):
    """
    Read all node times from files matching path_pattern, and compute time intervals
    such that each interval has equal number of node times.

    Parameters:
    - path_pattern: str, glob pattern to locate the input text files.
    - num_bins: int, number of equal-count bins (default 20)

    Returns:
    - T: np.ndarray of shape (num_bins+1,), where
         T[0] = 0, T[1]..T[num_bins-1] are computed from quantiles, and T[-1] = +inf
    """
    # Step 1: Read all node times
    all_times = []
    for file in glob.glob(path_pattern):
        with open(file, 'r') as f:
            times = [float(line.strip()) for line in f if line.strip()]
            all_times.extend(times)

    all_times = np.array(all_times)
    print(f'unique node time shape: {all_times.shape}')

    # Step 2: Compute quantile-based cutoffs
    quantile_cutoffs = np.quantile(all_times, np.linspace(0, 1, num_bins, endpoint=False)[1:])

    # Step 3: Add 0 at the beginning and +inf at the end
    T = np.concatenate(([0.0], quantile_cutoffs, [np.inf]))

    return T
T=get_equal_count_time_bins()
print('Dividing the time intervals based on Polegon nodes output\n', T)

unique node time shape: (2680900,)
Dividing the time intervals based on Polegon nodes output
 [    0.           203.28664636   396.75635352   635.8426295
   917.22988843  1235.31852792  1593.03277959  1993.74188566
  2438.60614208  2945.57167185  3532.62012737  4224.42830914
  5029.62089513  5977.51491856  7091.40570674  8442.37037801
 10119.98775789 12296.64936225 15294.64629212 19839.72323331
            inf]


In [10]:
# time interval depends on node time, each interval has equal amount of nodes

a_values = np.arange(0, n + 1)
b_values = np.arange(0,n + 1)
def safe_logsumexp(log_terms,signs=None):
    if len(log_terms) == 0:
        return -np.inf 
    max_x = np.max(log_terms) 
    scale=max_x
    if max_x<np.log(1.0e-40):
        scale=np.log(1.0e-40) 
    shifted_exps = np.exp(log_terms - scale)# Compute e^(-(x_i - min_x))
    if signs is not None:
        shifted_exps *= signs  # Apply sign adjustments if given
    inner_sum = np.sum(shifted_exps)  # Sum up scaled exponentials
    if inner_sum <= 0:
        return -np.inf  # Prevent log(0) issues
    log_inner_sum = np.log(inner_sum) + scale  # Compute final log sum exp
    return log_inner_sum

# Efficiently calculate rising factorial in log space
def log_rising_factorial(a, b):
    if b == 0 or a==0:
        return 0
    if a < b:
        return 0  
    return np.sum([np.log(a - i) for i in range(b)])

# Efficiently calculate falling factorial in log space
def log_falling_factorial(a, n):
    if n == 0 or a==0:
        return 0
    return np.sum([np.log(a + i) for i in range(n)])

# Generate dataframes for rising and falling factorials
def create_factorial_dataframes(a_values, b_values):
    rising_data = {
        f'a[{b}]': [log_rising_factorial(a, b) for a in a_values] for b in b_values
    }
    falling_data = {
        f'a_{n}': [log_falling_factorial(a, n) for a in a_values] for n in b_values
    }
    df_rising = pd.DataFrame(rising_data, index=a_values)
    df_falling = pd.DataFrame(falling_data, index=a_values)
    return df_rising, df_falling

# Function to calculate "choose 2 out of k" (k choose 2)
def choose_2_out_of_k(k):
    if k < 2:
        return 0
    return k*(k-1)/2
C1 = np.array([choose_2_out_of_k(k) for k in range(n + 1)])


df_rising, df_falling = create_factorial_dataframes(a_values, b_values)  



def logconstant_tables_Anp(n,Ne=1,t=1):
    signs_arr = np.empty((n + 1, n + 1), dtype=object)
    log_factorial=np.empty((n + 1, n + 1), dtype=object)
    exp_arr=np.empty((n + 1, n + 1), dtype=object)
    logbk=np.empty((n+1), dtype=object)
    ck=np.empty(n )
    for k_start in range(1,n+1):
        logbk_temp=[]
        for k_end in range(1,k_start+1):
            log_terms= []
            signs=[]
            exp_term=[]
            ck[k_end-1]=-C1[k_end]*t/2
            logbk_temp.append(df_rising.iloc[k_start, k_end]-df_falling.iloc[k_start, k_end]+np.log(2*k_end-1))
            for k in range(k_end, k_start + 1):
                # Numerator and denominator in log space
                numerator = (
                    np.log(2 * k - 1)
                    + df_falling.iloc[k_end, k - 1]
                    + df_rising.iloc[k_start, k]
                )

                denominator = (
                gammaln(k_end + 1)  # log(k_end!)
                + gammaln(k - k_end + 1)  # log((k - k_end)!)
                + df_falling.iloc[k_start, k]
                )
                signs.append(1-2*((k - k_end)%2))
                log_terms.append(numerator - denominator)
                exp_term.append(-C1[k]*t/2/Ne)

            signs_arr[k_start, k_end] = np.array(signs)
            log_factorial[k_start, k_end] = np.array(log_terms)
            exp_arr[k_start, k_end] = np.array(exp_term)
            logbk[k_start]=np.array(logbk_temp)
    return signs_arr, log_factorial,exp_arr,logbk,ck

signs_arr, log_factorial,exp_arr,logbk,ck = logconstant_tables_Anp(n)

def Ant_p(Ne,k_start, k_end,t):
    log_terms=exp_arr[k_start,k_end]*t/Ne+log_factorial[k_start,k_end]
    signs=signs_arr[k_start,k_end]
    # Combine positive and negative parts safely
    log_inner_sum = safe_logsumexp(log_terms,signs=signs)
    return log_inner_sum



def log_Postk_in_interval(data,T1,T2,kth,n,Ne):  
    log_terms= []
    k_end=n-kth+1
    used_time=data[(data <= T2) & (data > T1)]
    k_start=n
    for i in range(len(used_time)):
        t=used_time[i]-T1
        log_terms.append(Ant_p(Ne,k_start,k_end,t))        
    log_inner_sum = safe_logsumexp(log_terms)+ np.log((n-kth+1)*(n-kth)/4/Ne)
    
    return log_inner_sum

def dynamic_programming(n, Ne,T):
    t=np.array(T[1:-1]) - np.array(T[:-2])  # Time intervals
    num_interval = len(t)
    max_sum = n - 1

    # Initialize log_dp table: all -inf, except base case
    log_dp = np.full((max_sum + 1, num_interval), -np.inf)
    log_dp[0][0] = 0.0  # log(1) = 0

    # Fill log_dp table
    for i in range(1, num_interval):
        for s in range(max_sum + 1):
            terms = []
            for j in range(s + 1):
                log_prev = log_dp[s - j][i - 1]
                if np.isneginf(log_prev):
                    continue
                ant_val = Ant_p(Ne[i - 1], n - (s - j), n - s, t[i - 1])
                log_term = log_prev + ant_val
                terms.append(log_term)
            if terms:
                log_dp[s][i] = safe_logsumexp(np.array(terms))

    return log_dp


In [11]:
def log_posexp_first(data,T1,n,N_e1):
    log_Post_expectation=[]
    for i in range(1,n):#i+1=1,2.....19
        p_temp=log_Postk_in_interval(data[i-1,:],0,T1,i,n,N_e1)
        log_Post_expectation.append(p_temp)
    return np.array(log_Post_expectation)

def log_posexp_middle(data,T,n,Ne,dp):
    log_Post_total=[]
    num_iterval = len(T) - 2 # we are not considering the last intervals which goes to infinity
    for t_index in range(1,num_iterval):
        T1=T[t_index]
        T2=T[t_index+1]
        log_Post_expectation=[]
        for i in range(1,n):#i+1=1,2.....19
            log_P_total_i=[]
            for s in range(i):#s=0,1,2,....18
                prob_B_given_A = log_Postk_in_interval(data[i-1,:],T1,T2,i-s,n-s,Ne[t_index])
                prob_A_s = dp[s][t_index]                                   #i+1 here should also minus s>
                log_P_total_i.append(prob_A_s+prob_B_given_A)
            log_Post_expectation.append(log_P_total_i)
        log_Post_total.append(log_Post_expectation)
    return log_Post_total

def log_posexp_last(data,T2,n,N_elast,dp,T3=np.inf):
    log_Post_expectation=[]
    for i in range(1,n):
        log_P_total_i=[]
        for s in range(i):
            prob_A_s = dp[s][-1]
            prob_B_given_A = log_Postk_in_interval(data[i-1,:],T2,T3,i-s,n-s,N_elast)
            log_P_total_i.append(prob_A_s+prob_B_given_A)
        log_Post_expectation.append(log_P_total_i)
    return log_Post_expectation

def normalize_log_posterior(data,Ne,t,n):
    dp=dynamic_programming(n, Ne,t)
    log_posteriorfirst,log_posteriorlast=log_posexp_first(data,T[1],n,Ne[0]),log_posexp_last(data,T[-2],n,Ne[-1],dp)
    log_posteriormiddle=log_posexp_middle(data,T,n,Ne,dp)
    log_middle=[]
    # print(log_posterior)
    for i in range(len(log_posteriorfirst)):
        log_middle_i=np.array([row[i] if len(row) > i else None for row in log_posteriormiddle]).flatten().tolist()
        log_middlei=[row[i] if len(row) > i else None for row in log_posteriormiddle]
        constant=safe_logsumexp(np.array([log_posteriorfirst[i]]+log_middle_i+log_posteriorlast[i]))
        log_posteriorfirst[i]=log_posteriorfirst[i]-constant
        temp_middlei=[]
        for row in log_middlei:  
            temp_middlei.append(safe_logsumexp(np.array(row)) - constant)
        log_middle.append(temp_middlei)
        log_posteriorlast[i]=safe_logsumexp(log_posteriorlast[i])-constant
    log_middle=np.array(log_middle).T
    colfirst_pos=np.exp(log_posteriorfirst)
    colmiddle_pos=np.exp(log_middle)
    collast_pos=np.exp(log_posteriorlast)
    return colfirst_pos, colmiddle_pos, collast_pos

In [4]:
#input data, run only once
file_path = "/space/s1/KaiyuanLi/Ne_estimate/ratio1/1/arg50_1.trees"
ts = tskit.load(file_path)
# Path to the .trees file
num_tree=ts.num_trees
num_arg=100
span=[]
# Load the tree sequence
time=np.zeros([n-1,ts.num_trees,num_arg])
n2count=0
for j in range(1,2):
    file_path = f"/space/s1/KaiyuanLi/Ne_estimate/ratio1/1/arg50_{j+1}.trees"
    ts = tskit.load(file_path)
    # Iterate over each tree and find the 19th coalescence time
    for tree in ts.trees():
        coalescence_times = [
        ts.node(node).time for node in tree.nodes()
        if  ts.node(node).time > 0]
        # Sort times in ascending order
        coalescence_times.sort()
        if j==1:
            span.append(int(tree.interval[1] - tree.interval[0]))
        # print(tree.index)
        # Get the 19th coalescence time (if enough events exist)
       
                

# label the node and caluclate the node specific 
# some labeling of the arg.

In [12]:
# outdir='/space/s1/KaiyuanLi/Ne_estimate/ratio1/50'
# np.save(f"{outdir}/time.npy", time)
time=np.load('/space/s1/KaiyuanLi/Ne_estimate/ratio1/1/time.npy')

In [13]:
from scipy.optimize import minimize
# --- Precompute constants ---
def dp_to_p(dp, n):
    p_interval = np.zeros((n - 1,dp.shape[1]+1))
    convolution_p=np.zeros((n-1,dp.shape[1]))
    for i in range(1, n):
        for s in range(i, n):
            p_interval[i - 1,0] += np.exp(dp[s,1])
        for s in range(i):
            p_interval[i - 1,-1] += np.exp(dp[s,-1])

    # print(p_interval)
    convolution_p[:, 0] = p_interval[:, 0]
    for t_index in range(dp.shape[1]-1):
        for i in range(1, n):
            for s in range(i, n):
                convolution_p[i - 1,t_index] += np.exp(dp[s,t_index+1])
        p_interval[:, t_index]= convolution_p[:, t_index] - convolution_p[:, t_index - 1]
    return p_interval
# --- Partial objective function (Ne_last fixed) ---
def joint_objective_Ne_partial(x, T, subspan_rate, colfirst_pos,colmiddle_pos,collast_pos, n):
    Ne = x
    dp_temp = dynamic_programming(n, Ne,T)
    logp = dp_to_p(dp_temp, n)
    # print(np.sum(logp,axis=1))
    logp = np.where(np.isneginf(logp), -1e30, logp)
    # print(logp[0, 1:-1].shape)
    total = 0
    for j in range(len(subspan_rate)):
        for i in range(n-1):
            interval_loss = (
                colfirst_pos[j][i] * logp[i, 0] +
                np.sum(colmiddle_pos[j][:,i] * logp[i, 1:-1],axis=0) +
                collast_pos[j][i] * logp[i, -1]
            )
            total += subspan_rate[j] *interval_loss
        # print(total)
        
    return -total

# --- Ne3 estimator from posterior ---
def estimate_last_Ne( time, T):
    t_last = []
    for tree in range(time.shape[1]):
        for num_arg in range(time.shape[2]):
            for i in range(n-1):
                adjusted_values = time[i, tree, num_arg]- T
                if adjusted_values > 0:
                    t_last.extend([adjusted_values] * span[tree])
    return np.mean(t_last)

Ne_last =estimate_last_Ne(time,T[-2])

In [ ]:
# --- EM setup ---
ran_num = 10
x =800*np.ones(len(T)-2)
tolerance = len(T)-1
max_iter = 10
seed = 1
results = []
print("Start EM")
# Ne_last =estimate_last_Ne(time,T[-2])
for it in range(max_iter):
    random.seed(seed)
    selected_indices = random.sample(range(ts.num_trees), ran_num)
    subspan = np.array(span)[selected_indices]
    subspan_rate = subspan / np.sum(subspan)
    col1, col2, col3 = [], [], []
    for i in selected_indices:
        Ne = np.append(x, Ne_last)
        c1, c2, c3 = normalize_log_posterior(time[:, i, :],  Ne,T,n)
        col1.append(np.array(c1))
        col2.append(np.array(c2))
        col3.append(np.array(c3))
    res = minimize(
        lambda x: joint_objective_Ne_partial(x, T, subspan_rate, col1, col2, col3, n),
        x0=x,
        method='L-BFGS-B',
        bounds=[(300, 3000)] * len(x)
    )
    x_new = res.x
    loss_total = res.fun

    # Combine Ne vector for output
    full_Ne_new = np.append(x_new, Ne_last)

    print(f"[Iter {it + 1}] Ne = {np.round(full_Ne_new, 2)}")
    print(f"Loss: {loss_total:.2f}")

    results.append((*full_Ne_new, loss_total))

    # Convergence check
    delta = np.sum(np.abs(x - x_new))
    if delta < tolerance:
        break
    print('continue')
    # Update for next iteration
    x = x_new
    seed += 1


Start EM
[Iter 1] Ne = [1108.7   914.88 1159.39 3000.    664.25 2503.   1684.79 2020.37 1367.88
  742.32  446.18  300.    300.    582.12  759.3   804.14  794.71  800.
  800.   3607.  ]
Loss: -18.52
continue
[Iter 2] Ne = [ 976.33 1185.95  966.27 3000.    300.   3000.   1865.11 2885.42 3000.
  643.4  3000.    300.    300.    536.9   772.76  804.14  794.71  800.
  800.   3607.  ]
Loss: -19.88
continue
[Iter 3] Ne = [1207.35  901.94 2056.01 3000.    300.   3000.   3000.   3000.   3000.
 3000.   3000.    300.    300.    344.31  784.86  801.46  809.09  800.
  800.   3607.  ]
Loss: -19.49
continue
[Iter 4] Ne = [1491.94  999.89 2094.4  3000.    348.27 3000.   3000.   3000.   3000.
 3000.   3000.    300.    300.    341.51  784.91  801.6   809.09  800.
  800.   3607.  ]
Loss: -17.87
continue
[Iter 5] Ne = [1487.8  1136.43 3000.   3000.    316.89 3000.   3000.   3000.   3000.
 3000.   3000.    300.    300.    300.    786.43  799.83  807.36  800.
  800.   3607.  ]
Loss: -17.21
continue


In [15]:
# --- EM setup ---

for it in range(5,10):
    random.seed(seed)
    selected_indices = random.sample(range(ts.num_trees), ran_num)
    subspan = np.array(span)[selected_indices]
    subspan_rate = subspan / np.sum(subspan)
    col1, col2, col3 = [], [], []
    for i in selected_indices:
        Ne = np.append(x, Ne_last)
        c1, c2, c3 = normalize_log_posterior(time[:, i, :],  Ne,T,n)
        col1.append(np.array(c1))
        col2.append(np.array(c2))
        col3.append(np.array(c3))
    res = minimize(
        lambda x: joint_objective_Ne_partial(x, T, subspan_rate, col1, col2, col3, n),
        x0=x,
        method='L-BFGS-B',
        bounds=[(300, 3000)] * len(x)
    )
    x_new = res.x
    loss_total = res.fun

    # Combine Ne vector for output
    full_Ne_new = np.append(x_new, Ne_last)

    print(f"[Iter {it + 1}] Ne = {np.round(full_Ne_new, 2)}")
    print(f"Loss: {loss_total:.2f}")

    results.append((*full_Ne_new, loss_total))

    # Convergence check
    delta = np.sum(np.abs(x - x_new))
    if delta < tolerance:
        break
    print('continue')
    # Update for next iteration
    x = x_new
    seed += 1


[Iter 6] Ne = [1277.25  990.09 3000.   3000.    377.57 3000.   3000.   3000.   3000.
 3000.   3000.    300.    300.    300.    786.19  799.83  807.36  800.
  800.   3607.  ]
Loss: -17.64
continue
[Iter 7] Ne = [1467.46  647.02 2997.25 3000.    463.18 3000.   3000.   3000.   3000.
 3000.   3000.    300.    300.    300.    786.47  800.02  807.44  800.
  800.   3607.  ]
Loss: -18.14
continue
[Iter 8] Ne = [1643.3   890.87 3000.   3000.    378.26 3000.   3000.   3000.   3000.
 3000.   3000.    609.76  300.    300.    786.49  798.02  807.55  800.
  800.   3607.  ]
Loss: -16.99
continue
[Iter 9] Ne = [1309.53  829.54 3000.   3000.    514.82 3000.   3000.   3000.   3000.
 3000.   3000.    620.5   300.    300.    786.61  798.12  807.55  800.
  800.   3607.  ]
Loss: -18.56
continue
[Iter 10] Ne = [1241.9   864.68 3000.   3000.    433.16 3000.   3000.   3000.   3000.
 3000.   3000.   2303.46  300.    300.    805.92  810.12  807.11  800.
  800.   3607.  ]
Loss: -18.57
continue


In [16]:
# --- EM setup ---

for it in range(10,20):
    random.seed(seed)
    selected_indices = random.sample(range(ts.num_trees), ran_num)
    subspan = np.array(span)[selected_indices]
    subspan_rate = subspan / np.sum(subspan)
    col1, col2, col3 = [], [], []
    for i in selected_indices:
        Ne = np.append(x, Ne_last)
        c1, c2, c3 = normalize_log_posterior(time[:, i, :],  Ne,T,n)
        col1.append(np.array(c1))
        col2.append(np.array(c2))
        col3.append(np.array(c3))
    res = minimize(
        lambda x: joint_objective_Ne_partial(x, T, subspan_rate, col1, col2, col3, n),
        x0=x,
        method='L-BFGS-B',
        bounds=[(300, 3000)] * len(x)
    )
    x_new = res.x
    loss_total = res.fun

    # Combine Ne vector for output
    full_Ne_new = np.append(x_new, Ne_last)

    print(f"[Iter {it + 1}] Ne = {np.round(full_Ne_new, 2)}")
    print(f"Loss: {loss_total:.2f}")

    results.append((*full_Ne_new, loss_total))

    # Convergence check
    delta = np.sum(np.abs(x - x_new))
    if delta < tolerance:
        break
    print('continue')
    # Update for next iteration
    x = x_new
    seed += 1


[Iter 11] Ne = [1241.9   864.68 3000.   3000.    433.16 3000.   3000.   3000.   3000.
 3000.   3000.   2303.46  300.    300.    805.92  810.12  807.11  800.
  800.   3607.  ]
Loss: -18.91


In [ ]:
delta

0.0001733725566737121